# Data generation and preprocessing
## Generation
- Data is generated to simulate server metrics (CPU usage, memory usage, latency) over time, with incidents occurring randomly. Each incident lasts for a fixed duration and causes a spike in all metrics. See [data_gen.py](/scripts/data_gen.py)
## Preprocessing, Feature Engineering
To transform raw time-series metrics into a format suitable for supervised learning, we apply a sliding window approach. We label each sample as 1 if an incident occurs within the next H steps, and 0 otherwise. We extract the following features from the dataset
### Features
- For each server metric, we compute
    - long-term statistics (mean, std, min, max, trend) over the history window W
    - short-term statistics (mean, std) over the last 5 steps
    - divergence (short-term mean - long-term mean)
    - volatility shift (short-term std - long-term std)
    - 3 most recent raw values (t, t-1, t-2)
- We also create following interaction features:
    - CPU-latency interaction, Memory-latency interaction - captures how CPU/Memory load might amplify latency issues. This is taken into account, because a latency spike by itself could indicate a transient network blip, but if it coincides with high CPU or Memory usage, it is more likely to be a real incident.
    - Global stress index - represents total system load in window
    - Stress index trend - captures if the system load is accelerating

- See [data_preprocessing.py](/scripts/data_preprocessing.py)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from scripts.data_gen import generate_server_metrics
from scripts.data_preprocessing import build_dataset
from sklearn.ensemble import GradientBoostingClassifier
from scripts.evaluation import evaluate_test_set, print_metrics
import scripts.visualization as viz
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
N          = 6000   # total time steps
W          = 30     # history window size
H          = 10     # prediction horizon
INCIDENT_PROB   = 0.03   # ~3 % of steps start an incident
INCIDENT_LENGTH = 20     # each incident lasts 20 steps

df = generate_server_metrics(N, INCIDENT_PROB, INCIDENT_LENGTH)
print(f"Dataset: {len(df)} steps | incident steps: {df.incident.sum()} "
      f"({df.incident.mean()*100:.1f}%)")

metrics = ["cpu", "memory", "latency"]

X, y = build_dataset(df, W, H, metrics)
print(f"Samples: {len(X)} | positive (incident within H): " f"{y.sum()} ({y.mean()*100:.1f}%)")

feature_names = []
for m in metrics:
    feature_names += [
        f"{m}_mean_long",
        f"{m}_std_long",
        f"{m}_min",
        f"{m}_max",
        f"{m}_trend",
        f"{m}_divergence",
        f"{m}_vol_shift",
        f"{m}_val_t",
        f"{m}_val_t-1",
        f"{m}_val_t-2"
    ]
feature_names += ["Interaction_CPU_Lat", "Global_Stress_Mean", "Interaction_Mem_Lat", "Trend_Stress_Index"]

Dataset: 6000 steps | incident steps: 2082 (34.7%)
Samples: 5961 | positive (incident within H): 2872 (48.2%)


# Modeling
## Gradient Boosting Classifier (GBC) & SVM Ensemble
We employ a Voting Classifier that combines a Gradient Boosting Classifier (GBC) and a Support Vector Machine (SVM).

### Gradient Boosting Classifier (GBC)
- **Mechanism**: GBC constructs a predictive model by sequentially building multiple decision trees, where each new tree attempts to correct the residual errors of the previous ones.
- **Why it was used**: It is highly effective for tabular data derived from time-series windows, capturing complex, non-linear interactions between statistical features. GBC is generally robust to outliers and provides feature importance, which is useful for explaining alerts.
- **Limitations**: Can be sensitive to overfitting if the number of trees is too high without proper regularization (max_depth, subsample). It generally takes longer to train than simple linear models.

### Support Vector Machine (SVM)
- **Mechanism**: SVM finds the optimal hyperplane that maximizes the margin of separation between classes in a high-dimensional space, using the RBF kernel to handle non-linearity.
- **Why it was used**: We use SVM as a "second opinion" to GBC.
- **Limitations**: Computationally expensive ($O(n^2)$) as dataset size grows. Sensitive to noise and requires feature scaling (RobustScaler was used here).

### Voting Classifier (Ensemble)
- **Mechanism**: Averages the predicted probabilities from GBC and SVM (weighted 4:1) to make the final decision.
- **Why it was used**: Reduces individual model variance and bias. By combining the high-precision tendency of the GBC with the generalization capabilities of the SVM, we achieve a more robust decision boundary than either model could alone.
- **Limitations**: Inference is slower because both models must run. The final probability is a blend, making why a specific decision was reached slightly harder to trace back to a single rule.

## Evaluation
- **F1-Score**:
  *Why we watch this*: There is class imbalance in our dataset (incidents are somewhat rare). Accuracy is misleading (a model that never alerts is 97% accurate). F1 forces us to balance Precision and Recall (catching real incidents).
- **AUROC**:
  *Why we watch this*: Measures the model's ability to discriminate between "No incident" and "Incident" states across all possible thresholds. A score of 0.79 indicates good separability.
- **AUPRC**:
  *Why we watch this*: The Area Under the Precision-Recall Curve is the gold standard for imbalanced datasets. It focuses strictly on how well the model handles the positive class (incidents). A score of 0.87 means that when the model is confident an incident is happening, it is usually right.

### Final Results
```text
Best threshold: 0.5

==================================================
TEST SET RESULTS
==================================================
  AUROC            : 0.7803
  AUPRC            : 0.8677
  False Alarm Rate : 0.1584
  Mean Time-to-Alert: 9.2 steps ahead

  Confusion Matrix:
    TP=316  FP=61
    FN=194  TN=324
              precision    recall  f1-score   support

           0     0.6255    0.8416    0.7176       385
           1     0.8382    0.6196    0.7125       510

    accuracy                         0.7151       895
   macro avg     0.7318    0.7306    0.7151       895
weighted avg     0.7467    0.7151    0.7147       895
```


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from scripts.model import train_val_test_split, find_threshold_for_fixed_far
from scripts.evaluation import evaluate_test_set, print_metrics

X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(X, y)

svm_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('svc', SVC(probability=True, kernel='rbf', C=10, gamma='scale',
                class_weight='balanced', random_state=RANDOM_SEED))
])

gbc = GradientBoostingClassifier(
    n_estimators=600,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    random_state=RANDOM_SEED
)

voting_classifier = VotingClassifier(
    estimators=[('gb', gbc), ('svm', svm_pipeline)],
    voting='soft',
    weights=[4, 1]
)

voting_classifier.fit(X_train, y_train)

# Find the best threshold for a fixed False Alarm Rate (FAR), this is here because if we were finding the best threshold based on F1-score alone, we would end up with a very low threshold that would cause too many false alarms.
best_threshold = find_threshold_for_fixed_far(voting_classifier, X_val, y_val, max_far=0.15)
print(f"Best threshold: {best_threshold}")

# Evaluate with the adjusted threshold
test_result_metrics = evaluate_test_set(voting_classifier, X_test, y_test, best_threshold, H)
print_metrics(test_result_metrics)

y_pred = (voting_classifier.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)
print(classification_report(y_test, y_pred, digits=4))



Split  → train: 4172  val: 894  test: 895
Best threshold: 0.5

TEST SET RESULTS
  AUROC            : 0.7803
  AUPRC            : 0.8677
  False Alarm Rate : 0.1584
  Mean Time-to-Alert: 9.2 steps ahead

  Confusion Matrix:
    TP=316  FP=61
    FN=194  TN=324
              precision    recall  f1-score   support

           0     0.6255    0.8416    0.7176       385
           1     0.8382    0.6196    0.7125       510

    accuracy                         0.7151       895
   macro avg     0.7318    0.7306    0.7151       895
weighted avg     0.7467    0.7151    0.7147       895



# Visualization
- check dashboard below or [results](/results/report_final.png) for a cool dashboard that contains the graphs for the test set, feature importance, a snippet of predicted incidents and visualisation of the ROC and PR curves

# Results
### Key Strengths
**High Precision (~84%)**: When the model alerts, it is usually correct. This is critical for avoiding alert fatigue. The AUPRC of 0.87 further confirms that the model handles the class imbalance well.
**Early Detection**: The mean time-to-alert is 9.2 steps ahead, which could be translated to rougly 9 minutes. Given our prediction horizon was $H=10$, the model is detecting issues almost immediately as the leading indicators appear, giving operators ample time to react.
**Controlled False Alarm Rate**: By explicitly optimizing for a FAR of approx. 0.15, we successfully capped the noise level, preventing the system from flooding the dashboard with false positives.

### Limitations & Areas for Improvement
**Recall (~62%)**: The model currently misses roughly 40% of incidents (False Negatives). This indicates that certain failure modes might not be captured effectively by the current statistical features. Future iterations could explore more advanced features.
**Ensemble Diversity**: The voting classifier currently relies heavily on GBC (80% weight). Future iterations could incorporate a more diverse set of classifiers

![report_final](results/report_final.png "Report")

In [8]:
test_start_idx = len(df) - len(X_test)

fig_voting = viz.plot_full_dashboard(
    df=df,
    metric_cols=metrics,
    feature_names=feature_names,
    test_probs=voting_classifier.predict_proba(X_test)[:, 1],
    y_test=y_test,
    metrics=test_result_metrics,
    threshold=best_threshold,
    test_start_idx=test_start_idx,
    model=voting_classifier,
    W=W,
    H=H,
)
filename = lambda suffix: f"results/report{suffix}.png"
plt.savefig(filename("_final"),dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.close()
print("Saved report")

Saved report


#